In [1]:
import asyncio
import sys
from pathlib import Path
import pandas as pd
from datetime import datetime, timedelta
import time
import os

# Add the project root to the path to import core modules
notebook_path = Path().absolute()
project_root = notebook_path.parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from core.data_sources.clob import CLOBDataSource

async def fetch_historical_data(trading_pairs=['WLD-USDT', 'TRUMP-USDT'], 
                              intervals=['1s', '1m'], 
                              days=180):
    """Fetch historical candlestick data using existing CLOB infrastructure"""
    
    clob = CLOBDataSource()
    
    # Create output directory
    output_dir = Path('candlestick_data')
    output_dir.mkdir(exist_ok=True)
    
    for trading_pair in trading_pairs:
        for interval in intervals:
            try:
                print(f"Fetching {interval} data for {trading_pair}...")
                
                # Use existing get_candles_last_days method
                candles = await clob.get_candles_last_days(
                    connector_name="binance",
                    trading_pair=trading_pair,
                    interval=interval,
                    days=days
                )
                
                # Save to CSV
                filename = output_dir / f"{trading_pair.lower().replace('/', '_')}_{interval}.csv"
                candles.candles_df.to_csv(filename)
                print(f"Saved {filename}")
                
            except Exception as e:
                print(f"Error fetching {interval} data for {trading_pair}: {str(e)}")
                
    # Optionally dump the cache to disk
    clob.dump_candles_cache()

# For Jupyter notebook execution
async def run_fetch():
    await fetch_historical_data()
    print("Data collection complete!")

# Execute in notebook
await run_fetch()

Fetching 1s data for WLD-USDT...


CancelledError: 